# 01 - Orquestador: Pipeline Bronze → Silver → DQL → Gold

Trabajo Práctico 2 - Orquestación del flujo de extremo a extremo

## Objetivo

Coordinar, con `dbutils.notebook.run()`, la ejecución completa del pipeline en el orden correcto:

1. **Silver** (4 notebooks, independientes entre sí — se corren en paralelo): `02_Silver_RUES`, `03_Silver_SECOP`, `04_Silver_CIIU`, `05_Silver_TRM`. Cada uno lee su tabla Bronze, aplica sus reglas de calidad y escribe su tabla Silver + su tabla DQL correspondiente.
2. **Gold** (2 notebooks, con dependencia entre ellos — se corren en secuencia): `01_Gold_Empresas` (crea `gold.dim_empresa`, que es insumo del siguiente) y luego `02_Gold_Contratos` (hace `join` contra `gold.dim_empresa`).

Las rutas usan `dbutils.notebook.run(path, timeout, params)` con rutas **relativas** a la carpeta de este notebook (`../Trabajo Practico 2 - Capa Silver/...`, `../Trabajo Practico 2 - Capa Gold/...`), tal como está estructurado el repositorio. Para configurar esto como **Job de Databricks** con el grafo de dependencias visible: crear 6 tasks (una por notebook) + este notebook orquestador como task final, o usar directamente este notebook como la única task del Job (ejecuta todo internamente) — cualquiera de las dos formas cumple el requisito de "flujo organizado de dependencias ejecutándose en el orden correcto".

## Por qué no se puede probar el `dbutils.notebook.run()` fuera de Databricks

Este notebook se validó por partes: cada notebook Silver/Gold que invoca ya fue probado de forma independiente (localmente con pandas contra datos reales de Bronze, y en vivo con SQL equivalente contra el warehouse) antes de escribir este orquestador. Pero `dbutils.notebook.run()` solo existe en el runtime de Databricks — no hay forma de ejecutarlo fuera de un cluster/warehouse de Databricks, así que la ejecución real de este notebook debe hacerse en la UI de Databricks.

In [ ]:
dbutils.widgets.dropdown("modo", "paralelo", ["paralelo", "secuencial"], "Modo de ejecucion Silver")
dbutils.widgets.text("timeout_seg", "1800", "Timeout por notebook (segundos)")

MODO = dbutils.widgets.get("modo")
TIMEOUT = int(dbutils.widgets.get("timeout_seg"))

NOTEBOOKS_SILVER = [
    "../Trabajo Practico 2 - Capa Silver/02_Silver_RUES",
    "../Trabajo Practico 2 - Capa Silver/03_Silver_SECOP",
    "../Trabajo Practico 2 - Capa Silver/04_Silver_CIIU",
    "../Trabajo Practico 2 - Capa Silver/05_Silver_TRM",
]
NOTEBOOKS_GOLD = [
    "../Trabajo Practico 2 - Capa Gold/01_Gold_Empresas",
    "../Trabajo Practico 2 - Capa Gold/02_Gold_Contratos",
]

print(f"Modo Silver: {MODO} | Timeout por notebook: {TIMEOUT}s")

---

## Paso 1: Silver (RUES, SECOP, CIIU, TRM)

Son independientes entre sí (ninguno lee la salida de otro), así que en modo `paralelo` se lanzan los 4 al mismo tiempo con hilos (patrón estándar de `dbutils.notebook.run` concurrente); en modo `secuencial` se corren uno por uno, útil para depurar errores con trazas más claras.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def correr_notebook(path):
    inicio = time.time()
    try:
        resultado = dbutils.notebook.run(path, TIMEOUT)
        return (path, "OK", resultado, round(time.time() - inicio, 1))
    except Exception as e:
        return (path, "ERROR", str(e), round(time.time() - inicio, 1))


resultados_silver = []
if MODO == "paralelo":
    with ThreadPoolExecutor(max_workers=len(NOTEBOOKS_SILVER)) as executor:
        futuros = {executor.submit(correr_notebook, nb): nb for nb in NOTEBOOKS_SILVER}
        for futuro in as_completed(futuros):
            resultados_silver.append(futuro.result())
else:
    for nb in NOTEBOOKS_SILVER:
        resultados_silver.append(correr_notebook(nb))

for path, estado, detalle, segundos in resultados_silver:
    print(f"[{estado}] {path} ({segundos}s)")

fallos_silver = [r for r in resultados_silver if r[1] == "ERROR"]
if fallos_silver:
    for path, _, detalle, _ in fallos_silver:
        print(f"Detalle del error en {path}: {detalle}")
    raise Exception(f"{len(fallos_silver)} notebook(s) Silver fallaron -- no se continua a Gold")

print("\nTodos los notebooks Silver terminaron OK.")

---

## Paso 2: Gold (Empresas → Contratos, en ese orden)

`02_Gold_Contratos` lee `gold.dim_empresa`, que crea `01_Gold_Empresas` — por eso van en **secuencia**, nunca en paralelo. Si `01_Gold_Empresas` falla, no se corre `02_Gold_Contratos`.

In [ ]:
resultados_gold = []
for nb in NOTEBOOKS_GOLD:
    resultado = correr_notebook(nb)
    resultados_gold.append(resultado)
    path, estado, detalle, segundos = resultado
    print(f"[{estado}] {path} ({segundos}s)")
    if estado == "ERROR":
        print(f"Detalle del error: {detalle}")
        raise Exception(f"{path} fallo -- se detiene la cadena Gold (el siguiente notebook depende de este)")

print("\nTodos los notebooks Gold terminaron OK.")
print("\nPipeline completo: Bronze -> Silver -> DQL -> Gold ejecutado de extremo a extremo.")

---

## Siguiente paso

Continuar con `02_Prueba_Idempotencia.ipynb` (misma carpeta) para demostrar que correr este orquestador más de una vez no duplica ni altera los resultados.